In [1]:
"""
================================================================================
GARCH(1,1) WITH LIMIT-GAUSSIAN INNOVATIONS — MONTE CARLO SIMULATION STUDY
================================================================================

Compares finite-sample estimation performance of three estimators:
  1. Limit-Gaussian MLE  — correctly specified under sub-Gaussian DGPs
  2. Gaussian QMLE       — misspecified under all non-Gaussian DGPs
  3. Student-t MLE       — correctly specified under Student-t DGPs

Across three GARCH(1,1) persistence configurations (Cases A, B, C) and
seven innovation DGPs (sub-Gaussian nu=2,5,8; Student-t nu=5,8,12; Gaussian).

INPUTS:
  - GARCH_PARAMS: dict of true (omega, alpha, beta) for each persistence case
  - INNOVATIONS:  list of (innovation_type, nu) pairs defining the DGPs
  - T: time series length (default 1000)
  - N: number of Monte Carlo replications (default 100)

OUTPUTS:
  - all_results: list of dicts, one per replication, containing parameter
                 estimates, volatility RMSE, and stored series for plotting
  - summary_avg: pandas DataFrame of Monte Carlo averages by DGP/case
  - Printed tables: volatility RMSE, shape parameter estimates, and
                    parameter RMSE for omega, alpha, beta

RUNTIME: approximately 11 hours at T=1000, N=100 on a standard laptop.
         For a quick test, set T=200, N=5.

NOTE ON PARALLELISATION:
  joblib Parallel with loky backend is used for parallelisation across
  replications. np.random.seed() does not work reliably in loky workers;
  instead, a local numpy.random.default_rng is created per replication,
  with scipy t_rv.rvs called with random_state=rng to ensure independent
  and reproducible draws across parallel workers.

NOTE ON GRID SEARCH:
  For sub-Gaussian DGPs, the nu grid is centred around the true nu with
  proportional half-width +-50%, i.e. [0.5*nu*, 1.5*nu*]. This is a
  simulation study device motivated by the flatness of the log-likelihood
  surface in nu, which produces unreliable nu recovery under a wide fixed
  grid. For Gaussian and Student-t DGPs, a wide fixed grid [1.2, 25] is
  used. In practice, the true nu can be replaced by a method-of-moments
  estimate based on the sample kurtosis (see thesis appendix).
================================================================================
"""

import numpy as np
import pandas as pd
import time
import warnings
from scipy.stats import t as t_rv
from scipy.optimize import minimize
from scipy.special import hyp2f1, gammaln
from arch import arch_model
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# ── 1. UTILITIES ─────────────────────────────────────────────────────────────

def _reconstruct_sigma2(y, omega, alpha, beta_):
    """
    Reconstruct GARCH(1,1) conditional variance path from parameter estimates.
    
    Inputs:  y      — observed return series (array of length T)
             omega  — GARCH intercept (> 0)
             alpha  — ARCH coefficient (>= 0)
             beta_  — GARCH coefficient (>= 0)
    Output:  s2     — estimated conditional variance path (array of length T)
    
    Initialises at unconditional variance omega/(1-alpha-beta) if stationary,
    otherwise at sample variance. Applies floor of 1e-12 for numerical stability.
    """
    T = len(y)
    s2 = np.zeros(T, float)
    s2[0] = (omega / max(1e-12, 1 - alpha - beta_)) if (alpha + beta_ < 0.999) else np.var(y)
    for t in range(1, T):
        s2[t] = max(omega + alpha * y[t-1]**2 + beta_ * s2[t-1], 1e-12)
    return s2


# ── 2. LIMIT-GAUSSIAN DISTRIBUTION ───────────────────────────────────────────

def subgaussian_eps(T, nu, rng):
    """
    Generate T standardised limit-Gaussian innovations with tail index nu.
    
    The limit-Gaussian distribution arises as the weak limit of Y/(1+Y^2/nu)
    where Y ~ Student-t(nu). Standardised to unit variance via scale factor
    s_e = nu / sqrt((nu+1)(nu+3)).
    
    Inputs:  T   — number of innovations to generate
             nu  — tail index (> 1); smaller nu = lighter tails
             rng — numpy.random.Generator instance for reproducibility
    Output:  standardised limit-Gaussian innovations (array of length T)
    """
    Y = t_rv.rvs(df=nu, size=T, random_state=rng)
    X = Y / (1 + (Y**2)/nu)
    s_e = nu / np.sqrt((nu+1)*(nu+3))
    return X / s_e


def sub_mle_negloglik_fixed_nu(garch_theta, y, nu, penalty_scale=1e10):
    """
    Negative log-likelihood for GARCH(1,1) with limit-Gaussian innovations,
    evaluated at fixed nu. Used as the objective function in the grid search.
    
    Inputs:  garch_theta   — (omega, alpha, beta) parameter vector
             y             — observed return series
             nu            — fixed tail index value
             penalty_scale — penalty for boundary violations (default 1e10)
    Output:  negative log-likelihood value (scalar)
    
    Observations with scaled residuals outside the support boundary
    |z_t| >= sqrt(nu)/2 are handled via a soft quadratic penalty rather
    than hard exclusion, ensuring a continuous objective function.
    """
    omega, alpha, beta_ = garch_theta
    
    # Enforce parameter constraints via penalty
    if (omega <= 1e-6) or (alpha < 0) or (beta_ < 0) or (alpha + beta_ >= 0.999):
        return penalty_scale
    
    sigma2 = _reconstruct_sigma2(y, omega, alpha, beta_)
    
    # Compute scaled residuals and check support boundary
    s_e = nu / np.sqrt((nu+1)*(nu+3))
    scaled_z = (y / np.sqrt(sigma2)) * s_e
    limit = np.sqrt(nu) / 2
    
    outside = np.abs(scaled_z) >= limit
    if outside.any():
        dist = np.maximum(0.0, np.abs(scaled_z[outside]) - limit + 1e-6)
        penalty = 1e4 * np.sum(dist**2)
    else:
        penalty = 0.0
    
    safe_z = scaled_z[~outside]
    if len(safe_z) == 0:
        return penalty_scale
    
    # Evaluate log-density via hypergeometric function (vectorised z_arg, loop over hyp2f1)
    z_args = np.maximum(1 - 4*safe_z**2/nu, 1e-10)
    log_norm = (gammaln((nu+1)/2) - 0.5*np.log(np.pi) - gammaln(nu/2)
                - 0.5*np.log(nu) + ((3-nu)/2)*np.log(2))
    
    log_vals = np.empty(len(safe_z))
    for i, (z, za) in enumerate(zip(safe_z, z_args)):
        try:
            val = hyp2f1((3-nu)/4, (1-nu)/4, 0.5, za)
            log_vals[i] = (log_norm - 0.5*np.log(za) + np.log(val)
                          if (val > 0 and np.isfinite(val)) else -1e6)
        except:
            log_vals[i] = -1e6
    
    ll = np.sum(log_vals)
    total_ll = ll - penalty + len(y)*np.log(s_e) - 0.5*np.sum(np.log(sigma2))
    
    return -total_ll if np.isfinite(total_ll) else penalty_scale


# ── 3. ESTIMATION WRAPPERS ────────────────────────────────────────────────────

def estimate_qmle(y):
    """
    Estimate GARCH(1,1) parameters via Gaussian QMLE using the arch package.
    
    Input:  y — observed return series
    Output: [omega, alpha, beta] estimates, or [nan, nan, nan] on failure
    """
    try:
        model = arch_model(y, mean='zero', vol='GARCH', p=1, q=1, dist='normal')
        res = model.fit(disp='off', show_warning=False)
        return [res.params['omega'], res.params['alpha[1]'], res.params['beta[1]']]
    except:
        return [np.nan]*3


def estimate_sub_grid_mle(y, nu_true=None):
    """
    Estimate GARCH(1,1) parameters via limit-Gaussian MLE using a grid search
    over nu, with QMLE warm start and multiple restarts per grid point.
    
    Input:  y        — observed return series
            nu_true  — if provided, grid is centred at [0.5*nu_true, 1.5*nu_true]
                       (simulation study device); if None, wide grid [1.2, 25] used
    Output: [omega, alpha, beta, nu_hat] — best estimates across all grid points
            and restarts; [nan]*4 if no successful optimisation
    
    For each nu in the grid, SLSQP minimises the negative log-likelihood over
    (omega, alpha, beta) subject to stationarity constraint alpha+beta < 1.
    Gaussian QMLE estimates provide the warm start; two random restarts per
    grid point mitigate local optima sensitivity.
    """
    var_y = np.var(y)

    # Build nu grid
    if nu_true is not None:
        lo = max(1.2, nu_true * 0.5)
        hi = nu_true * 1.5
        nu_grid = np.linspace(lo, hi, 15)
    else:
        nu_grid = np.linspace(1.2, 25, 15)

    best_ll = -np.inf
    best_params = [np.nan] * 4
    constraints = {'type': 'ineq', 'fun': lambda x: 0.999 - x[1] - x[2]}
    bounds = [(1e-6, None), (1e-6, 0.999), (1e-6, 0.999)]

    # QMLE warm start
    qmle = estimate_qmle(y)
    base_theta = [qmle[0], qmle[1], qmle[2]] if np.isfinite(qmle[0]) else [0.02*var_y, 0.10, 0.85]

    for n in nu_grid:
        for r in range(2):  # 2 restarts per grid point
            theta0 = np.array(base_theta, float)
            theta0 *= np.random.uniform(0.8, 1.2, size=3)  # jitter around warm start
            theta0[1] = np.clip(theta0[1], 1e-4, 0.3)
            theta0[2] = np.clip(theta0[2], 0.5, 0.95)

            res = minimize(
                sub_mle_negloglik_fixed_nu, theta0, args=(y, n),
                method='SLSQP', bounds=bounds, constraints=constraints,
                options={'ftol': 1e-6, 'maxiter': 200}
            )
            if res.success and -res.fun > best_ll:
                best_ll = -res.fun
                best_params = [res.x[0], res.x[1], res.x[2], n]

    return best_params


def estimate_t_mle(y):
    """
    Estimate GARCH(1,1) parameters via Student-t MLE using the arch package.
    
    Input:  y — observed return series
    Output: [omega, alpha, beta, nu] estimates, or [nan]*4 on failure
    """
    try:
        model = arch_model(y, mean='zero', vol='GARCH', p=1, q=1, dist='t')
        res = model.fit(disp='off', show_warning=False)
        return [res.params['omega'], res.params['alpha[1]'], res.params['beta[1]'], res.params['nu']]
    except:
        return [np.nan]*4


# ── 4. SIMULATION PARAMETERS ──────────────────────────────────────────────────

# True GARCH(1,1) parameters for three persistence cases
# Case A: high persistence (alpha+beta=0.90)
# Case B: very high persistence (alpha+beta=0.95)
# Case C: moderate persistence (alpha+beta=0.80)
GARCH_PARAMS = {
    'A': (0.10, 0.10, 0.80),
    'B': (0.05, 0.05, 0.90),
    'C': (0.20, 0.20, 0.60)
}

# Innovation DGPs: (type, nu) pairs
# Sub-Gaussian nu=2,5,8: limit-Gaussian innovations, lighter than Gaussian
# t nu=5,8,12: Student-t innovations, heavier than Gaussian
# normal: Gaussian innovations
INNOVATIONS = [
    ('Sub-Gaussian', 2.0), ('Sub-Gaussian', 5.0), ('Sub-Gaussian', 8.0),
    ('t', 5.0), ('t', 8.0), ('t', 12.0),
    ('normal', None)
]

T = 1000   # time series length
N = 100    # number of Monte Carlo replications


# ── 5. SINGLE REPLICATION FUNCTION ───────────────────────────────────────────

def run_one_rep(rep, itype, iparam, omega, alpha, beta_, T):
    """
    Run a single Monte Carlo replication.
    
    Inputs:  rep    — replication index (used to seed the RNG)
             itype  — innovation type ('Sub-Gaussian', 't', or 'normal')
             iparam — innovation parameter (nu for sub-Gaussian/t; None for normal)
             omega, alpha, beta_ — true GARCH parameters
             T      — time series length
    Output:  dict containing parameter estimates, squared errors, volatility RMSE,
             stored y series and true sigma2 path for all three estimators
    
    RNG note: a local numpy.random.default_rng is seeded per replication to
    ensure reproducibility across parallel workers. scipy t_rv.rvs is called
    with random_state=rng to avoid the global state issue with joblib loky.
    """
    # Seed local RNG — unique per replication and innovation configuration
    rng = np.random.default_rng(rep * 1000 + hash((itype, str(iparam))) % 10000)

    # Generate innovations
    if itype == 'Sub-Gaussian':
        eps = subgaussian_eps(T, iparam, rng)
    elif itype == 't':
        eps = t_rv.rvs(df=iparam, size=T, random_state=rng)
        eps /= np.sqrt(iparam / (iparam - 2))  # standardise to unit variance
    else:
        eps = rng.normal(0, 1, T)

    # Simulate GARCH(1,1) process: y_t = sigma_t * eps_t
    y = np.zeros(T)
    sigma2_true = np.zeros(T)
    sigma2_true[0] = omega / (1 - alpha - beta_)  # initialise at unconditional variance
    for t in range(1, T):
        sigma2_true[t] = omega + alpha * y[t-1]**2 + beta_ * sigma2_true[t-1]
        y[t] = np.sqrt(sigma2_true[t]) * eps[t]

    # Estimate under all three specifications
    nu_for_grid = iparam if itype == 'Sub-Gaussian' else None
    res_sub = estimate_sub_grid_mle(y, nu_true=nu_for_grid)
    
    # Filter implausible sub-Gaussian estimates (omega > 50x sample variance)
    var_y = np.var(y)
    if np.isfinite(res_sub[0]) and res_sub[0] > 50 * var_y:
        res_sub = [np.nan] * 4

    res_q = estimate_qmle(y)
    res_t = estimate_t_mle(y)

    # Compute volatility RMSE for each estimator
    def vol_rmse(params):
        if np.all(np.isfinite(params[:3])):
            s2_hat = _reconstruct_sigma2(y, params[0], params[1], params[2])
            return np.sqrt(np.mean((np.sqrt(sigma2_true) - np.sqrt(s2_hat))**2))
        return np.nan

    return {
        'case': None,         # stamped in main loop
        'rep': rep,
        'innov_type': itype,
        'nu_true': iparam,
        # Parameter estimates
        'omega_sub': res_sub[0], 'alpha_sub': res_sub[1],
        'beta_sub':  res_sub[2], 'nu_sub':   res_sub[3],
        'omega_t':   res_t[0],  'alpha_t':   res_t[1],
        'beta_t':    res_t[2],  'nu_t':      res_t[3],
        'omega_qmle': res_q[0], 'alpha_qmle': res_q[1], 'beta_qmle': res_q[2],
        # Volatility RMSE
        'vol_rmse_sub': vol_rmse(res_sub),
        'vol_rmse_q':   vol_rmse(res_q),
        'vol_rmse_t':   vol_rmse(res_t),
        # Squared errors for RMSE tables
        'se_omega_sub': (res_sub[0]-omega)**2 if np.isfinite(res_sub[0]) else np.nan,
        'se_alpha_sub': (res_sub[1]-alpha)**2 if np.isfinite(res_sub[1]) else np.nan,
        'se_beta_sub':  (res_sub[2]-beta_)**2 if np.isfinite(res_sub[2]) else np.nan,
        'se_omega_q':   (res_q[0]-omega)**2   if np.isfinite(res_q[0])   else np.nan,
        'se_alpha_q':   (res_q[1]-alpha)**2   if np.isfinite(res_q[1])   else np.nan,
        'se_beta_q':    (res_q[2]-beta_)**2   if np.isfinite(res_q[2])   else np.nan,
        'se_omega_t':   (res_t[0]-omega)**2   if np.isfinite(res_t[0])   else np.nan,
        'se_alpha_t':   (res_t[1]-alpha)**2   if np.isfinite(res_t[1])   else np.nan,
        'se_beta_t':    (res_t[2]-beta_)**2   if np.isfinite(res_t[2])   else np.nan,
        # Stored series for representative plot
        'y_series':     y.tolist(),
        'sigma2_true':  sigma2_true.tolist(),
        'omega_true': omega, 'alpha_true': alpha, 'beta_true': beta_,
    }


# ── 6. MAIN SIMULATION LOOP ───────────────────────────────────────────────────

# Parallelise over replications using joblib loky backend
# Each (case, innovation) combination runs N replications in parallel
all_results = []
start_time = time.time()

for case, (omega, alpha, beta_) in GARCH_PARAMS.items():
    for itype, iparam in INNOVATIONS:
        print(f"Running Case {case} | Innov: {itype} {iparam}")

        reps = Parallel(n_jobs=-1, backend='loky')(
            delayed(run_one_rep)(rep, itype, iparam, omega, alpha, beta_, T)
            for rep in range(N)
        )

        for r in reps:
            r['case'] = case  # stamp case label (cannot close over in parallel function)

        all_results.extend(reps)
        print(f"  Done ({(time.time()-start_time)/60:.1f} min elapsed)")


# ── 7. RESULTS TABLES ─────────────────────────────────────────────────────────

# Aggregate results: Monte Carlo averages by (case, innovation type, nu)
df = pd.DataFrame(all_results)
summary_avg = df.groupby(['case', 'innov_type', 'nu_true'], dropna=False).apply(
    lambda x: x.select_dtypes(include='number').apply(np.nanmean)
)

# Table 1: Volatility RMSE and shape parameter recovery
print("\n" + "="*60)
print("TABLE 1: VOLATILITY RMSE & SHAPE PARAMETER (nu)")
print("="*60)
print(summary_avg[['nu_sub', 'nu_t', 'vol_rmse_sub', 'vol_rmse_q', 'vol_rmse_t']])

# Table 2: Mean parameter estimates (bias assessment)
print("\n" + "="*60)
print("TABLE 2: MEAN PARAMETER ESTIMATES (omega, alpha, beta)")
print("="*60)
print(summary_avg[['omega_sub', 'omega_qmle', 'omega_t',
                   'alpha_sub', 'alpha_qmle', 'alpha_t',
                   'beta_sub',  'beta_qmle',  'beta_t']])

# Tables 3-5: Parameter RMSE
for param, col_suffix, label in [
    ('omega', 'w', 'OMEGA'), ('alpha', 'a', 'ALPHA'), ('beta', 'b', 'BETA')
]:
    print("\n" + "="*60)
    print(f"TABLE: RMSE FOR {label}")
    print("="*60)
    summary_avg[f'RMSE_{col_suffix}_sub'] = np.sqrt(summary_avg[f'se_{param}_sub'])
    summary_avg[f'RMSE_{col_suffix}_q']   = np.sqrt(summary_avg[f'se_{param}_q'])
    summary_avg[f'RMSE_{col_suffix}_t']   = np.sqrt(summary_avg[f'se_{param}_t'])
    print(summary_avg[[f'RMSE_{col_suffix}_sub', f'RMSE_{col_suffix}_q', f'RMSE_{col_suffix}_t']])

elapsed = (time.time() - start_time) / 60
print(f"\nSimulation complete in {elapsed:.2f} minutes.")

Running Case A | Innov: Sub-Gaussian 2.0
  Done (9.2 min elapsed)
Running Case A | Innov: Sub-Gaussian 5.0
  Done (20.0 min elapsed)
Running Case A | Innov: Sub-Gaussian 8.0
  Done (34.3 min elapsed)
Running Case A | Innov: t 5.0
  Done (87.6 min elapsed)
Running Case A | Innov: t 8.0
  Done (136.1 min elapsed)
Running Case A | Innov: t 12.0
  Done (182.3 min elapsed)
Running Case A | Innov: normal None
  Done (228.1 min elapsed)
Running Case B | Innov: Sub-Gaussian 2.0
  Done (237.6 min elapsed)
Running Case B | Innov: Sub-Gaussian 5.0
  Done (246.3 min elapsed)
Running Case B | Innov: Sub-Gaussian 8.0
  Done (259.1 min elapsed)
Running Case B | Innov: t 5.0
  Done (297.6 min elapsed)
Running Case B | Innov: t 8.0
  Done (332.4 min elapsed)
Running Case B | Innov: t 12.0
  Done (369.1 min elapsed)
Running Case B | Innov: normal None
  Done (400.6 min elapsed)
Running Case C | Innov: Sub-Gaussian 2.0
  Done (408.9 min elapsed)
Running Case C | Innov: Sub-Gaussian 5.0
  Done (418.6 min 

In [17]:
"""
================================================================================
REPRESENTATIVE VOLATILITY PANEL — FIGURE
================================================================================

Generates a three-panel figure comparing estimated conditional volatility paths
against the true path for a single representative replication, under three
innovation DGPs: limit-Gaussian (nu=2), Gaussian, and Student-t (nu=12).
All panels use Case A (high persistence, alpha+beta=0.90).

INPUTS:
  - all_results    : list of dicts from the main simulation loop
  - GARCH_PARAMS   : dict of true GARCH parameters (from simulation cell)
  - _reconstruct_sigma2 : volatility reconstruction function (from simulation cell)
  - N              : number of replications (from simulation cell)

OUTPUT:
  - representative_volatility_panel_caseA_finalsim.pdf
    Three-panel figure saved to working directory. Each panel plots the true
    conditional volatility path sigma_t against estimated paths from all three
    estimators for a single replication.

NOTE:
  A random seed (2024) is used to select the representative replication
  reproducibly. If the selected replication has a failed sub-Gaussian estimate,
  the code falls back to the first replication with a valid estimate.
================================================================================
"""

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('PDF')  # use PDF backend for high-quality output

# ── Select representative replication ────────────────────────────────────────

np.random.seed(2024)                        # fixed seed for reproducibility
fix_rep = np.random.randint(0, N)           # randomly selected replication index

# DGPs and persistence case for the panel
case = 'A'
omega, alpha, beta_ = GARCH_PARAMS[case]

panel_targets = [
    ('Sub-Gaussian', 2.0),   # panel 1: limit-Gaussian DGP, lightest tails
    ('normal',       None),  # panel 2: Gaussian DGP
    ('t',            12.0),  # panel 3: Student-t DGP, lightest heavy-tailed spec
]

panel_titles = [
    r'Limit-Gaussian Innovations ($\nu^* = 2.0$)',
    r'Gaussian Innovations',
    r'Student-$t$ Innovations ($\nu^* = 12.0$)',
]

# ── Build figure ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for ax, (itype, iparam), title in zip(axes, panel_targets, panel_titles):

    # Retrieve the selected replication from all_results
    # For normal DGP, nu_true is stored as None/NaN so match on innov_type only
    if iparam is None:
        matches = [r for r in all_results
                   if r['case'] == case
                   and r['innov_type'] == 'normal'
                   and r['rep'] == fix_rep
                   and np.all(np.isfinite([r['omega_sub'], r['alpha_sub'], r['beta_sub']]))]
    else:
        matches = [r for r in all_results
                   if r['case'] == case
                   and r['innov_type'] == itype
                   and r['nu_true'] == iparam
                   and r['rep'] == fix_rep
                   and np.all(np.isfinite([r['omega_sub'], r['alpha_sub'], r['beta_sub']]))]

    # Fallback: if selected rep has failed sub-Gaussian estimate, use first valid rep
    if not matches:
        if iparam is None:
            matches = [r for r in all_results
                       if r['case'] == case
                       and r['innov_type'] == 'normal'
                       and np.all(np.isfinite([r['omega_sub'], r['alpha_sub'], r['beta_sub']]))]
        else:
            matches = [r for r in all_results
                       if r['case'] == case
                       and r['innov_type'] == itype
                       and r['nu_true'] == iparam
                       and np.all(np.isfinite([r['omega_sub'], r['alpha_sub'], r['beta_sub']]))]
        if not matches:
            print(f"No valid replication found for {itype} {iparam} — skipping panel.")
            continue

    rec = matches[0]
    used_rep = rec['rep']

    # Retrieve stored series and reconstruct volatility paths
    y          = np.asarray(rec['y_series'],   float)
    sigma2_true = np.asarray(rec['sigma2_true'], float)
    vol_true   = np.sqrt(np.maximum(sigma2_true, 1e-12))

    # Reconstruct estimated conditional variance for each estimator
    vol_sub = np.sqrt(np.maximum(
        _reconstruct_sigma2(y, rec['omega_sub'],  rec['alpha_sub'],  rec['beta_sub']),  1e-12))
    vol_q   = np.sqrt(np.maximum(
        _reconstruct_sigma2(y, rec['omega_qmle'], rec['alpha_qmle'], rec['beta_qmle']), 1e-12))
    vol_t   = np.sqrt(np.maximum(
        _reconstruct_sigma2(y, rec['omega_t'],    rec['alpha_t'],    rec['beta_t']),    1e-12))

    nu_sub_est = rec['nu_sub']
    nu_t_est   = rec['nu_t']
    tgrid      = np.arange(len(y))

    # Plot volatility paths
    ax.plot(tgrid, vol_true, label=r'True $\sigma_t$',
            color='black', lw=1.5, alpha=0.6)
    ax.plot(tgrid, vol_sub,
            label=f'Limit-Gaussian MLE ($\\hat{{\\nu}}={nu_sub_est:.1f}$)',
            color='blue', lw=0.9)
    ax.plot(tgrid, vol_q,
            label='Gaussian QMLE',
            color='red', lw=0.9, alpha=0.8)
    ax.plot(tgrid, vol_t,
            label=f'Student-$t$ MLE ($\\hat{{\\nu}}={nu_t_est:.1f}$)',
            color='green', lw=0.9, alpha=0.8)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', ncols=2, fontsize=9, frameon=False)
    ax.grid(True, alpha=0.3)

# Shared axis labels
axes[1].set_ylabel(r'Conditional Volatility $\sigma_t$', fontsize=12)
axes[2].set_xlabel('Time $t$', fontsize=12)

plt.tight_layout()
plt.savefig('representative_volatility_panel_caseA_finalsim.pdf', bbox_inches='tight')
plt.close()

print(f"Saved: representative_volatility_panel_caseA_finalsim.pdf")
print(f"Seed=2024, selected replication index={fix_rep}")

Saved representative panel (seed=2024, initial rep=8)


In [18]:
"""
================================================================================
SINGLE-REPLICATION DIAGNOSTICS
================================================================================

Prints a summary table of parameter estimates, persistence, and volatility RMSE
for the representative replication used in the volatility panel figure, across
all three innovation DGPs and all three estimators.

INPUTS:
  - all_results    : list of dicts from the main simulation loop
  - panel_targets  : list of (itype, iparam) DGP specifications (from plot cell)
  - panel_titles   : list of panel title strings (from plot cell)
  - case           : persistence case label (from plot cell)
  - used_rep       : replication index used in the panel (from plot cell)
  - omega, alpha, beta_ : true GARCH parameters for the case (from plot cell)
  - _reconstruct_sigma2 : volatility reconstruction function (from simulation cell)

OUTPUT:
  Printed table for each DGP panel showing:
    - True parameter values and persistence (alpha* + beta*)
    - For each estimator: omega_hat, alpha_hat, beta_hat, persistence, vol_RMSE
  Complements the visual analysis in the representative panel figure.
================================================================================
"""

print(f"\nDiagnostics for representative panel (Case {case}, rep={used_rep})")
print("="*80)

for (itype, iparam), title in zip(panel_targets, panel_titles):

    # Retrieve the record for this DGP and the representative replication
    # Normal DGP has nu_true stored as None/NaN so match on innov_type only
    if iparam is None:
        recs = [r for r in all_results
                if r['case'] == case
                and r['innov_type'] == 'normal'
                and r['rep'] == used_rep]
    else:
        recs = [r for r in all_results
                if r['case'] == case
                and r['innov_type'] == itype
                and r['nu_true'] == iparam
                and r['rep'] == used_rep]

    if not recs:
        print(f"\n{title}: no record found for rep={used_rep}")
        continue

    rec         = recs[0]
    y           = np.asarray(rec['y_series'],    float)
    sigma2_true = np.asarray(rec['sigma2_true'], float)
    vol_true    = np.sqrt(np.maximum(sigma2_true, 1e-12))

    # Print panel header
    print(f"\n{title}")
    print("-"*80)
    print(f"{'Estimator':<25} {'omega_hat':>10} {'alpha_hat':>10} {'beta_hat':>10} "
          f"{'persistence':>12} {'vol_RMSE':>10}")
    print("-"*80)

    # True parameter row for reference
    true_persistence = alpha + beta_
    print(f"{'True parameters':<25} {omega:>10.4f} {alpha:>10.4f} {beta_:>10.4f} "
          f"{true_persistence:>12.4f} {'—':>10}")

    # Estimator rows
    estimators = [
        ('Limit-Gaussian MLE', rec['omega_sub'],  rec['alpha_sub'],  rec['beta_sub'],  rec['nu_sub']),
        ('Gaussian QMLE',      rec['omega_qmle'], rec['alpha_qmle'], rec['beta_qmle'], None),
        ('Student-t MLE',      rec['omega_t'],    rec['alpha_t'],    rec['beta_t'],    rec['nu_t']),
    ]

    for name, om, al, be, nu_est in estimators:

        # Handle failed estimates
        if not np.all(np.isfinite([om, al, be])):
            print(f"{name:<25} {'NaN':>10} {'NaN':>10} {'NaN':>10} {'NaN':>12} {'NaN':>10}")
            continue

        # Compute persistence and volatility RMSE for this estimator
        persistence = al + be
        s2_hat  = _reconstruct_sigma2(y, om, al, be)
        vol_hat = np.sqrt(np.maximum(s2_hat, 1e-12))
        vol_rmse_val = np.sqrt(np.mean((vol_true - vol_hat)**2))

        # Append estimated nu to estimator name where applicable
        nu_str = f" (ν̂={nu_est:.1f})" if nu_est is not None and np.isfinite(nu_est) else ""
        label  = name + nu_str

        print(f"{label:<25} {om:>10.4f} {al:>10.4f} {be:>10.4f} "
              f"{persistence:>12.4f} {vol_rmse_val:>10.4f}")

print("="*80)
print(f"\nTrue GARCH parameters: omega*={omega}, alpha*={alpha}, beta*={beta_}, "
      f"persistence alpha*+beta*={alpha+beta_}")


Diagnostics for representative panel (Case A, rep=8)

Sub-Gaussian Innovations ($\nu_{true} = 2.0$)
-----------------------------------------------------------------
Estimator            omega_hat  alpha_hat   beta_hat  persistence   vol_RMSE
-----------------------------------------------------------------
True parameters          0.1000    0.1000     0.8000       0.9000          —
Sub-Gaussian MLE(ν̂=2.7)     0.1198     0.1088     0.7602       0.8690     0.0243
Gaussian QMLE            0.1451     0.0776     0.7724       0.8500     0.0199
Student-t MLE(ν̂=500.0)     0.1454     0.0786     0.7719       0.8504     0.0190

Gaussian Innovations
-----------------------------------------------------------------
Estimator            omega_hat  alpha_hat   beta_hat  persistence   vol_RMSE
-----------------------------------------------------------------
True parameters          0.1000    0.1000     0.8000       0.9000          —
Sub-Gaussian MLE(ν̂=21.6)     0.3203     0.2481     0.6228      

In [1]:
"""
================================================================================
MONTE CARLO DIAGNOSTICS — BIAS, MCSD, AND RMSE
================================================================================

Computes and prints Monte Carlo bias, standard deviation (MCSD), and RMSE
for each GARCH parameter (omega, alpha, beta) across all estimators, DGPs,
and persistence cases, using the full set of N=100 replications.

INPUTS:
  - all_results : list of dicts from the main simulation loop, each containing
                  parameter estimates and true parameter values per replication
  - df_results  : pandas DataFrame constructed from all_results

OUTPUTS:
  Printed tables for:
    - Bias:  theta_bar - theta*, where theta_bar = (1/N) sum theta_hat^(r)
    - MCSD:  Monte Carlo standard deviation across replications
    - RMSE:  sqrt((1/N) sum (theta_hat^(r) - theta*)^2) = sqrt(Bias^2 + MCSD^2)

NOTE ON COVERAGE AND SE RATIOS:
  Standard error estimation via the observed information matrix is not
  asymptotically valid for the limit-Gaussian MLE, as the parameter-dependent
  support violates standard MLE regularity conditions. Wald-based coverage
  probabilities and SE ratios are therefore not reported. MCSD provides a
  simulation-based alternative measure of estimator precision that does not
  rely on asymptotic normality.
================================================================================
"""

# ── Bias / MCSD / RMSE computation ───────────────────────────────────────────

def compute_bias_mcsd_rmse(df):
    """
    Compute Monte Carlo bias, MCSD, and RMSE for each parameter, estimator,
    and DGP configuration.

    Input:  df — pandas DataFrame with one row per replication, containing
                 parameter estimates (omega_sub, alpha_sub, etc.) and true
                 values (omega_true, alpha_true, beta_true)
    Output: pandas DataFrame with columns:
                 case, innov_type, nu_true, estimator, parameter,
                 bias, mcsd, rmse, theta_bar, true_val, n_reps
    """
    rows = []
    for (case, itype, nu_true), grp in df.groupby(
            ['case', 'innov_type', 'nu_true'], dropna=False):

        # Read true parameter values from stored columns
        omega_true = grp['omega_true'].iloc[0]
        alpha_true = grp['alpha_true'].iloc[0]
        beta_true  = grp['beta_true'].iloc[0]

        for est, cols in [
            ('Limit-Gaussian', ('omega_sub',  'alpha_sub',  'beta_sub')),
            ('QMLE',           ('omega_qmle', 'alpha_qmle', 'beta_qmle')),
            ('t-MLE',          ('omega_t',    'alpha_t',    'beta_t')),
        ]:
            for param, col, true_val in zip(
                ['omega', 'alpha', 'beta'], cols,
                [omega_true, alpha_true, beta_true]
            ):
                vals = pd.to_numeric(grp[col], errors='coerce').dropna()
                if vals.empty:
                    continue

                theta_bar = vals.mean()
                bias      = theta_bar - true_val           # Monte Carlo bias
                mcsd      = vals.std(ddof=1)               # Monte Carlo std dev
                rmse      = np.sqrt(np.mean((vals - true_val)**2))  # RMSE

                rows.append({
                    'case': case, 'innov_type': itype, 'nu_true': nu_true,
                    'estimator': est, 'parameter': param,
                    'bias': bias, 'mcsd': mcsd, 'rmse': rmse,
                    'theta_bar': theta_bar, 'true_val': true_val,
                    'n_reps': len(vals)
                })

    return pd.DataFrame(rows)


# ── Run diagnostics ───────────────────────────────────────────────────────────

df_results = pd.DataFrame(all_results)
pd.set_option('display.max_rows',    None)
pd.set_option('display.max_columns', None)

bias_table = compute_bias_mcsd_rmse(df_results)

# Replace NaN nu_true with 'normal' for display
bias_table['nu_true'] = bias_table['nu_true'].fillna('normal')

# Print bias table
print("BIAS TABLE")
print("="*70)
print(bias_table.pivot_table(
    index=['case', 'innov_type', 'nu_true', 'parameter'],
    columns='estimator',
    values='bias'
).round(4))

# Print MCSD table
print("\nMCSD TABLE")
print("="*70)
print(bias_table.pivot_table(
    index=['case', 'innov_type', 'nu_true', 'parameter'],
    columns='estimator',
    values='mcsd'
).round(4))

# Print RMSE table
print("\nRMSE TABLE")
print("="*70)
print(bias_table.pivot_table(
    index=['case', 'innov_type', 'nu_true', 'parameter'],
    columns='estimator',
    values='rmse'
).round(4))



NameError: name 'pd' is not defined